In [ ]:
%pip install holidays
%pip install tensorflow
%pip install scikit-learn
%pip install pandas
%pip install numpy
%pip install lets-plot


import pandas as pd
import numpy as np
import holidays

# Make numpy values easier to read.
np.set_printoptions(precision=3, suppress=True)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score

import matplotlib.pyplot as plt
from lets_plot import *
LetsPlot.setup_html()

In [ ]:
bikes = pd.read_csv(
    "https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv")

bikes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112475 entries, 0 to 112474
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   dteday        112475 non-null  object 
 1   hr            112475 non-null  float64
 2   casual        112475 non-null  int64  
 3   registered    112475 non-null  int64  
 4   temp_c        112475 non-null  float64
 5   feels_like_c  112475 non-null  float64
 6   hum           112475 non-null  float64
 7   windspeed     112475 non-null  float64
 8   weathersit    112475 non-null  int64  
 9   season        112475 non-null  int64  
 10  holiday       112475 non-null  int64  
 11  workingday    112475 non-null  int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 10.3+ MB


## Make some features

In [ ]:
def create_features(df):
    # holidays
    df['dteday'] = pd.to_datetime(df['dteday'])
    dc_holidays = holidays.US(state='DC')
    df['holiday_name'] = df['dteday'].apply(lambda x: dc_holidays.get(x))
    holiday_features = pd.get_dummies(df['holiday_name'], dtype=int)
    df = pd.concat([df, holiday_features], axis=1)

    # hour of day cycles
    df["hour_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hr"] / 24)


    # month cycle
    df['month'] = df['dteday'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # day cycle
    df['day_of_week'] = df['dteday'].dt.dayofweek.astype(int)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)




    return df.drop(columns=['holiday_name', 'month', 'day_of_week'])

In [ ]:
df = create_features(bikes)

df.info()

df.describe().transpose()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112475 entries, 0 to 112474
Data columns (total 39 columns):
 #   Column                                                   Non-Null Count   Dtype         
---  ------                                                   --------------   -----         
 0   dteday                                                   112475 non-null  datetime64[ns]
 1   hr                                                       112475 non-null  float64       
 2   casual                                                   112475 non-null  int64         
 3   registered                                               112475 non-null  int64         
 4   temp_c                                                   112475 non-null  float64       
 5   feels_like_c                                             112475 non-null  float64       
 6   hum                                                      112475 non-null  float64       
 7   windspeed                             

,count,mean,min,25%,50%,75%,max,std
dteday,112475,2017-06-01 00:13:39.638142208,2011-01-01 00:00:00,2014-03-17 00:00:00,2017-06-01 00:00:00,2020-08-16 00:00:00,2023-10-31 00:00:00,NaN
hr,112475.0,11.501098,0.0,6.0,12.0,18.0,23.0,6.921864
casual,112475.0,90.434612,0.0,7.0,36.0,122.0,1244.0,128.655621
registered,112475.0,249.193625,0.0,48.0,180.0,360.0,1702.0,258.267544
temp_c,112475.0,15.376487,-14.7,7.6,16.0,23.5,40.5,9.749467
feels_like_c,112475.0,14.659325,-24.0,5.4,16.0,23.5,49.6,11.428324
hum,112475.0,0.636624,0.0889,0.4841,0.6409,0.7988,1.0,0.190328
windspeed,112475.0,13.100614,0.0,7.7,12.2,17.5,69.8,7.8576
weathersit,112475.0,1.405441,1.0,1.0,1.0,2.0,4.0,0.68345
season,112475.0,2.495799,1.0,2.0,2.0,3.0,4.0,1.101152


## Train

In [ ]:
df = df.sort_values('dteday')

df = df.drop('dteday', axis=1)

split_idx = int(len(df) * 0.8)

X_train = df.iloc[:split_idx]
X_test = df.iloc[split_idx:]

y_train = X_train[['casual', 'registered']].copy()
y_test = X_test[['casual', 'registered']].copy()

X_train = X_train.drop(columns=['casual', 'registered'])
X_test = X_test.drop(columns=['casual', 'registered'])

In [ ]:
norm = MinMaxScaler().fit(X_train)

X_train = norm.transform(X_train)

X_test = norm.transform(X_test)

# Set how much training data to hold out for validation.
val_size = int(len(X_train) * 0.25)
# Shuffle the training data each epoch.
buffer_size = len(X_train)
ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(
    buffer_size, reshuffle_each_iteration=True
)
# Batch and prefetch so training runs faster.
train_ds = ds.skip(val_size).batch(8224).prefetch(tf.data.AUTOTUNE)
val_ds = ds.take(val_size).batch(8224).prefetch(tf.data.AUTOTUNE)

# Build a batched test dataset for predictions.
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(8224).prefetch(tf.data.AUTOTUNE)

In [ ]:
model = Sequential()
model.add(Dense(128, input_dim=len(X_train[0]), activation='relu'))
model.add(Dropout(.25))
model.add(Dense(256, activation='swish'))
model.add(Dropout(.25))
model.add(Dense(64, activation='swish'))

model.add(Dense(2, activation='linear'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 128)            │         4,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,338 (212.26 KB)

 Trainable params: 54,338 (212.26 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
opt = keras.optimizers.Adam()
model.compile(loss="mean_squared_error", optimizer=opt, metrics=['mse'])

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_mse', patience=30)

history = model.fit(train_ds, epochs=2000, validation_data=val_ds, callbacks=[early_stop])

hist = pd.DataFrame(history.history)

Epoch 1/2000
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 244ms/step - loss: 73118.7500 - mse: 73118.7500 - val_loss: 73070.1953 - val_mse: 73070.1953
Epoch 2/2000
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 70572.3438 - mse: 70572.3438 - val_loss: 66313.8359 - val_mse: 66313.8359
Epoch 3/2000
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 60436.1328 - mse: 60436.1328 - val_loss: 49798.0898 - val_mse: 49798.0898
Epoch 4/2000
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 45448.0625 - mse: 45448.0625 - val_loss: 42409.6836 - val_mse: 42409.6836
Epoch 5/2000
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - loss: 41166.4570 - mse: 41166.4570 - val_loss: 40606.7617 - val_mse: 40606.7617
Epoch 6/2000
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - loss: 38877.7969 - mse: 38877.7969 - val_loss: 37738.4062 - val_mse: 37738.4062
Epoch 7/2000
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 37045.3867 - mse: 37045.3867 - val_loss: 35697.5977 - val_mse: 35697.5977
Epoch 8/2000
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step -

In [ ]:
hist = hist.reset_index()

hist_long = pd.melt(
    hist,
    id_vars='index',
    value_vars=['mse', 'val_mse'],
    var_name='series',
    value_name='mse_value'
)

(ggplot(hist_long, aes(x='index', y='mse_value', color='series'))
 + geom_line()
 + ggsize(700, 500)
 + theme_minimal()
)

In [ ]:
predictions = np.round(model.predict(test_ds), 1)
predictions

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step


array([[  0. ,   6.9],
       [  0. ,  10.1],
       [  0. ,  20.3],
       ...,
       [  3.6,  23.9],
       [ 68.7, 230.4],
       [  8.4,  77.9]], dtype=float32)

In [ ]:
result = root_mean_squared_error(y_test, predictions)
result

120.32279968261719

In [ ]:
r2 = r2_score(y_test,predictions)
r2

0.5847907066345215

In [ ]:
pred = pd.DataFrame(predictions, columns=['casual_pred', 'registered_pred'])
pred['casual_actual'] = y_test['casual'].to_numpy()
pred['registered_actual'] = y_test['registered'].to_numpy()
pred['casual_diff'] = pred['casual_actual'] - pred['casual_pred']
pred['registered_diff'] = pred['registered_actual'] - pred['registered_pred']

(ggplot(pred, aes(x='casual_actual', y='casual_pred'))
 + geom_point()
 + geom_abline(slope=1, intercept=0, color='red')
 + ggsize(700, 500)
 + theme_minimal()
)